![interpreto_banner](../assets/img/interpreto_banner.png)

# Generation Concept-based Explanation Tutorial

Welcome to this tutorial, our will be to obtain concept-based explanations starting from the beginning.

There are five key steps for concepts based explanations:

1. [**Split** your model in two parts](#split)
2. [Compute a dataset of **activations**](#activations)
3. [**Fit** a concept model on activations](#fit)
4. [Find the globally **important** concepts](#important)
5. [**Interpret** the concept dimensions](#interpret)

On which we add a bonus step:

6. [**Locally** important concepts](#locally)
7. [**Evaluate** concept-based explanations](#evaluate)

*Author: Antonin Poché*

## 1. **Split** your model in two parts <a class="anchor" id="split"></a>

Load the model and list modules to find where to split it.

In [1]:
from transformers import AutoModelForMaskedLM, AutoTokenizer

model = AutoModelForMaskedLM.from_pretrained("EuroBERT/EuroBERT-210m", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("EuroBERT/EuroBERT-210m")
split_point = "model.layers.10.mlp"

print(list(model.named_children()))

[('model', EuroBertModel(
  (embed_tokens): Embedding(128256, 768, padding_idx=128001)
  (layers): ModuleList(
    (0-11): 12 x EuroBertDecoderLayer(
      (self_attn): EuroBertAttention(
        (q_proj): Linear(in_features=768, out_features=768, bias=False)
        (k_proj): Linear(in_features=768, out_features=768, bias=False)
        (v_proj): Linear(in_features=768, out_features=768, bias=False)
        (o_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (mlp): EuroBertMLP(
        (gate_proj): Linear(in_features=768, out_features=3072, bias=False)
        (up_proj): Linear(in_features=768, out_features=3072, bias=False)
        (down_proj): Linear(in_features=3072, out_features=768, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): EuroBertRMSNorm((768,), eps=1e-05)
      (post_attention_layernorm): EuroBertRMSNorm((768,), eps=1e-05)
    )
  )
  (norm): EuroBertRMSNorm((768,), eps=1e-05)
  (rotary_emb): EuroBertRotaryEmbedding()
)), (

### Split the model using the `ModelWithSplitPoints` class

In [2]:
from interpreto import ModelWithSplitPoints

splitted_model = ModelWithSplitPoints(
    model_or_repo_id=model,
    tokenizer=tokenizer,
    split_points=split_point,
    device_map="cuda",
    batch_size=64,
)

## 2. Compute a datasets of **activations** <a class="anchor" id="activations"></a>

In [3]:
from datasets import load_dataset

rotten_tomatoes = load_dataset("cornell-movie-review-data/rotten_tomatoes")["train"]["text"]

WORD_GRANULARITY = ModelWithSplitPoints.activation_granularities.WORD

activations = splitted_model.get_activations(
    rotten_tomatoes,
    activation_granularity=WORD_GRANULARITY,
)

print(activations[split_point].shape)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


torch.Size([187890, 768])


## 3. **Fit** a concept model on activations <a class="anchor" id="fit"></a>

In [4]:
from interpreto.concepts import ICAConcepts

concept_explainer = ICAConcepts(splitted_model, nb_concepts=50)

concept_explainer.fit(activations)

## 4. Find the globally **important** concepts <a class="anchor" id="important"></a>

In [ ]:
# TODO

## 5. **Interpret** the concept dimensions <a class="anchor" id="interpret"></a>

In [5]:
from interpreto.concepts.interpretations import TopKInputs

topk_words_method = TopKInputs(
    concept_explainer=concept_explainer,
    activation_granularity=WORD_GRANULARITY,
    k=10,
)

interpretations = topk_words_method.interpret(
    inputs=rotten_tomatoes,
    latent_activations=activations,
    concepts_indices="all",
)

In [6]:
for concept_id, words_importance in interpretations.items():
    print(f"Concept {concept_id}: {list(words_importance.keys()) if words_importance is not None else 'None'}")

Concept 0: [' makes', ' make', ' made', ' lifts', ' leaves', ' making', ' sent', ' render', ' leave', ' keep']
Concept 1: [' to', ' will', ' and', ' about', ' much', ' with', ' of', ' out', ' very', ' some']
Concept 2: [' as', ' so', ' such', 'so', ' that', 'as', ' too', ' like', ' how', 'such']
Concept 3: [' more', ' less', 'more', ' better', ' greater', ' most', ' fewer', ' easier', ' longer', ' cleaner']
Concept 4: [' see', ' watch', ' through', ' missing', ' all', ' catch', ' every', ' seen', ' gets', ' displaying']
Concept 5: [' a', ' the', ' ', ' his', ' my', ' an', ' her', ' la', ' its', ' awful']
Concept 6: ['-fashioned', ' simplistic', ' lesser', ' primitive', ' had', ' inferior', ' mundane', '-made', ' just', ' disparate']
Concept 7: [' latest', ' moments', ' single', ' next', ' another', ' sometimes', ' if', ' moment', ' one', ' last']
Concept 8: [' has', ' had', ' and', '.', 'odd', ' poo', 'cinematic', 'a', ' important', ' film']
Concept 9: [' to', ' into', '-to', ' toward'

In [7]:
import os

from interpreto.concepts.interpretations import LLMLabels
from interpreto.model_wrapping.llm_interface import OpenAILLM

llm_labels_method = LLMLabels(
    concept_explainer=concept_explainer,
    llm_interface=OpenAILLM(api_key=os.environ["OPENAI_API_KEY"]),
    activation_granularity=LLMLabels.activation_granularities.SAMPLE,
    sampling_method=LLMLabels.sampling_methods.TOP,
    k_context=3,
)

interpretations = concept_explainer.interpret(
    LLMLabels,
    concepts_indices=[0, 1, 2],
    llm_interface=OpenAILLM(api_key=os.environ["OPENAI_API_KEY"]),
    inputs=rotten_tomatoes[:20],
)

AttributeError: type object 'LLMLabels' has no attribute 'activation_granularities'

In [ ]:
for concept_id, label in interpretations.items():
    print(f"Concept {concept_id}: {label}")

## 6. **Locally** important concepts <a class="anchor" id="locally"></a>

In [ ]:
# TODO

## 7. **Evaluate** concept-based explanations <a class="anchor" id="evaluate"></a>

### 7.1 Evaluate the concept-space from the [third part](#fit)

### 7.2 Evaluate the concepts-interpretations from the [fifth step](#important)